# Week 12: A Language Model Predicts the Next Token

This notebook follows the reviewed Week 12 presentation. Read each concept and calculate the worked example before running its code.

## Lesson map

1. A Language Model Predicts the Next Token
2. Tokenization Converts Text into IDs
3. Embeddings Turn Token IDs into Vectors
4. Self-Attention Mixes Information Across Tokens
5. The Context Window Is a Finite Token Budget
6. Logits Become Probabilities through Softmax
7. Sampling Controls How a Token Is Chosen
8. Base and Instruction Models Have Different Training Goals
9. Hallucination Is Unsupported Generation
10. Quantization Trades Precision for Resource Use
11. Ollama Provides a Local Model Runtime
12. Guided Lab: Trace One Generated Token

Use the same reasoning loop throughout: **predict, run, inspect, explain**.


## 1. A Language Model Predicts the Next Token

A **language model** assigns probabilities to possible token sequences.

An autoregressive large language model repeatedly predicts:

`P(next token | tokens already in context)`

Generation is a loop:

1. tokenize the input;
2. calculate next-token scores;
3. convert scores to probabilities;
4. choose one token;
5. append it to context;
6. repeat until a stopping condition.

The model does not retrieve truth automatically. It predicts plausible continuations from learned parameters and supplied context.

### Work it out first

Context: `The capital of Sri Lanka is`

Candidate probabilities might include:

`Colombo 0.55`, `Sri 0.12`, `Kandy 0.08`, others `0.25`.

The selected token is appended and the model predicts again.

### Notebook bridge

The LLMs 101 notebook performs a simple next-token prediction before chat generation.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
prompt = "The capital of Sri Lanka is"
outputs = model.generate(**tokenizer(prompt, return_tensors="pt"), max_new_tokens=5)

Expected output:

```text
A sequence containing the prompt tokens followed by generated tokens.
```


## 2. Tokenization Converts Text into IDs

A **tokenizer** converts text into tokens and integer **token IDs** from a fixed vocabulary.

A token may be:

- a complete word;
- part of a word;
- punctuation;
- whitespace pattern;
- special control marker.

Tokenization depends on the model. Character count and word count do not equal token count.

The decoder maps token IDs back to text, but decoding individual tokens can look unusual because boundaries and spaces are encoded.

### Work it out first

The word `unhelpful` might become pieces such as:

`un` + `help` + `ful`

If these map to IDs `[320, 912, 77]`, the model receives numbers, not raw letters.

### Notebook bridge

The notebook inspects token IDs and decodes generated sequences.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
encoded = tokenizer("Data science", add_special_tokens=False)
print(encoded["input_ids"])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

Expected output:

```text
A model-specific list of token IDs and token strings.
```


## 3. Embeddings Turn Token IDs into Vectors

An **embedding** maps a token ID to a learned vector.

If hidden size is `d`, each token begins as a vector with `d` numerical components. The model also needs position information because token order changes meaning.

During transformer layers, representations become **contextual**: the vector for a token changes according to surrounding tokens.

Embedding similarity can reflect learned usage patterns, but it does not guarantee factual or logical equivalence.

### Work it out first

Sequence length `6`, hidden size `4`.

Token-ID shape: `(6,)`  
Embedding shape: `(6,4)`

There is one four-number vector per token.

### Notebook bridge

RAG later uses separate embedding models to represent document chunks and queries.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
input_ids = torch.tensor([[10, 20, 30]])
embeddings = model.get_input_embeddings()(input_ids)
print(embeddings.shape)

Expected output:

```text
torch.Size([1, 3, hidden_size])
```


## 4. Self-Attention Mixes Information Across Tokens

A **transformer** processes token representations with repeated attention and feed-forward blocks.

In **self-attention**, each token:

1. forms a query describing what information it seeks;
2. compares that query with keys from allowed tokens;
3. converts comparisons into attention weights;
4. combines value vectors using those weights.

In causal generation, a token cannot attend to future tokens that have not been generated.

### Work it out first

Sentence: `The bank raised its rate because it expected inflation.`

When representing the second `it`, attention may assign more weight to `bank` and surrounding context. The result is a context-dependent representation.

### Notebook bridge

The notebook uses a pretrained transformer; learners now understand its core data flow.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
attention(Q, K, V) = softmax(QK^T / sqrt(dk)) V

Expected output:

```text
One context-mixed vector per token for each attention head.
```


## 5. The Context Window Is a Finite Token Budget

The **context window** is the maximum token sequence the model can process in one request, including some combination of:

- system and user messages;
- conversation history;
- retrieved documents;
- tool results;
- generated output.

Longer context consumes memory, latency, and cost. Content may be truncated or rejected when the limit is exceeded.

More context is not always better: irrelevant or conflicting text can reduce answer quality.

### Work it out first

Context limit `8,000` tokens.

Instructions `500`  
History `2,000`  
Retrieved evidence `3,500`  
Reserved output `1,000`

Total `7,000`, leaving `1,000` tokens of headroom.

### Notebook bridge

Learners should inspect prompt token counts before generation.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
token_count = len(tokenizer(prompt)["input_ids"])
print(token_count)

Expected output:

```text
The model-specific number of input tokens.
```


## 6. Logits Become Probabilities through Softmax

For each vocabulary token, the model produces a **logit**: an unrestricted score.

Softmax converts logits into probabilities:

`P(i) = e^(zi) / Σj e^(zj)`

- `zi`: logit for token `i`
- exponentials make values positive
- denominator adds exponentials for all candidates
- probabilities sum to `1`

Only relative logit differences matter.

### Work it out first

Logits `[2,1,0]`

Exponentials approximately `[7.39,2.72,1.00]`  
Sum `11.11`

Probabilities:

`[7.39/11.11, 2.72/11.11, 1/11.11]`  
`≈ [0.665,0.245,0.090]`

### Notebook bridge

The notebook inspects model logits and selects likely next tokens.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
logits = torch.tensor([2.0, 1.0, 0.0])
print(torch.softmax(logits, dim=0))

Expected output:

```text
tensor([0.6652, 0.2447, 0.0900])
```


## 7. Sampling Controls How a Token Is Chosen

**Greedy decoding** selects the highest-probability token each step.

**Sampling** randomly selects according to a probability distribution.

**Temperature** rescales logits before softmax:

`softmax(logits / T)`

- lower `T` sharpens the distribution;
- higher `T` flattens it;
- temperature does not add knowledge or verify facts.

Top-k and top-p restrict which candidates may be sampled.

### Work it out first

At low temperature, probabilities might be `[0.90,0.08,0.02]`.  
At higher temperature, they might be `[0.55,0.30,0.15]`.

The first remains most likely, but alternatives are selected more often.

### Notebook bridge

Learners compare greedy and sampled generation while keeping the prompt fixed.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
outputs = model.generate(
    **inputs,
    do_sample=True,
    temperature=0.7,
    max_new_tokens=40,
)

Expected output:

```text
One sampled continuation whose exact wording can differ between runs.
```


## 8. Base and Instruction Models Have Different Training Goals

A **base model** is pretrained mainly to continue token sequences.

An **instruction model** is further adapted using instruction-response examples and often preference training so it follows conversational tasks more reliably.

Instruction models usually expect a model-specific **chat template** containing role and control tokens. Sending plain text to an instruction model or chat-formatted text to a base model changes behaviour.

Neither model type guarantees truth, safety, or task correctness.

### Work it out first

Prompt to a base model:

`Question: What is 2+2? Answer:`

may continue a learned document pattern.

An instruction model receives structured messages such as user request and assistant turn, then generates the assistant response.

### Notebook bridge

The LLMs 101 notebook loads both base and instruction-tuned variants.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
messages = [{"role": "user", "content": "Explain tokenization simply."}]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

Expected output:

```text
A model-specific text prompt containing the required chat control tokens.
```


## 9. Hallucination Is Unsupported Generation

A **hallucination** is generated content presented as factual despite lacking support or being incorrect.

LLMs can:

- invent citations;
- combine incompatible facts;
- follow false assumptions in a prompt;
- produce outdated information;
- make arithmetic or logical mistakes;
- sound confident while uncertain.

Mitigations include trusted retrieval, tools, structured validation, explicit refusal rules, test datasets, and human review. A prompt alone cannot guarantee factuality.

### Work it out first

Question asks for a policy section that does not exist in the supplied document.

Unsafe answer invents a section number.  
Grounded answer states that the evidence does not contain the requested policy and cites what was checked.

### Notebook bridge

Later RAG lessons add evidence retrieval and citation checks around generation.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
if not supporting_chunks:
    return {"answer": None, "reason": "insufficient_evidence"}

Expected output:

```text
A machine-readable refusal instead of an unsupported factual answer.
```


## 10. Quantization Trades Precision for Resource Use

**Quantization** stores or computes model values with lower numerical precision.

Possible effects:

- lower memory use;
- faster inference on supported hardware;
- smaller model files;
- some quality loss;
- hardware- and implementation-dependent speed.

Quantization does not reduce prompt tokens or expand the context window automatically. Evaluate the quantized model on the actual tasks before deployment.

### Work it out first

One billion parameters:

- 16-bit storage is roughly `2 GB` for raw weights;
- 8-bit storage is roughly `1 GB`;
- 4-bit storage is roughly `0.5 GB`.

Runtime requires additional memory for cache and application state.

### Notebook bridge

Ollama commonly runs quantized local models; learners should inspect the selected model size.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
Approximate raw weight bytes = parameter_count x bits_per_parameter / 8

Expected output:

```text
1,000,000,000 x 4 / 8 = approximately 500,000,000 bytes.
```


## 11. Ollama Provides a Local Model Runtime

Ollama runs supported models locally and exposes a command-line and HTTP interface.

Key ideas:

- model identifier and version determine the artifact;
- local runtime still consumes CPU, memory, and storage;
- prompts and responses remain on the machine unless the application sends data elsewhere;
- local does not automatically mean secure;
- output must still be validated and evaluated.

Use only model licences and sizes suitable for the intended environment.

### Work it out first

Workflow:

1. pull a small model;
2. run one prompt;
3. inspect token and latency behaviour;
4. call the local API;
5. stop the model when not needed.

### Notebook bridge

The Ollama quickstart notebook introduces local generation after the model mechanism is understood.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
ollama run <model-name> "Explain what a token is in one sentence."

Expected output:

```text
A locally generated response from the selected installed model.
```


## 12. Guided Lab: Trace One Generated Token

Complete:

1. tokenize a short prompt and inspect IDs;
2. report input token count;
3. inspect model and tokenizer identifiers;
4. obtain next-token logits;
5. apply softmax to a small candidate set;
6. identify the highest-probability token;
7. compare greedy and sampled generation;
8. change temperature and describe the distribution change;
9. compare base and instruction prompt formats;
10. run a small local model with Ollama;
11. record latency and model size;
12. document one unsupported answer and a safer response.

### Work it out first

For logits `[2,1,0]`, probabilities are approximately `[0.665,0.245,0.090]`. Greedy decoding chooses the first token; sampling may choose another.

### Notebook bridge

Complete `05.llms-101.ipynb` and `28.ollama-quickstart.ipynb`.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
next_logits = outputs.logits[0, -1]
top = torch.topk(torch.softmax(next_logits, dim=-1), k=5)
print(top.indices, top.values)

Expected output:

```text
Five model-specific token IDs and their next-token probabilities.
```


## Guided lab

Complete:

1. tokenize a short prompt and inspect IDs;
2. report input token count;
3. inspect model and tokenizer identifiers;
4. obtain next-token logits;
5. apply softmax to a small candidate set;
6. identify the highest-probability token;
7. compare greedy and sampled generation;
8. change temperature and describe the distribution change;
9. compare base and instruction prompt formats;
10. run a small local model with Ollama;
11. record latency and model size;
12. document one unsupported answer and a safer response.

### Reference result

For logits `[2,1,0]`, probabilities are approximately `[0.665,0.245,0.090]`. Greedy decoding chooses the first token; sampling may choose another.


In [ ]:
# Guided lab workspace: Week 12
# Add only the imports needed for the current step.

# TODO 1: Prepare the smallest valid input.

# TODO 2: Apply the concept taught in this lesson.

# TODO 3: Display inspectable intermediate evidence.

# TODO 4: Compare the result with a hand calculation or stated requirement.

## Weekly deliverable

Submit the completed guided lab with:

- your prediction before execution;
- intermediate values, shapes, metrics, or traces;
- one failed assumption and its correction;
- a plain-English explanation of the result;
- the source notebook section you are now ready to complete.


## Sources and source notebooks

- <https://huggingface.co/docs/transformers/main_classes/text_generation>
- <https://github.com/curiousily/AI-Bootcamp/blob/master/05.llms-101.ipynb>
- <https://huggingface.co/docs/transformers/tokenizer_summary>
- <https://huggingface.co/docs/transformers/main_classes/output>
- <https://jalammar.github.io/illustrated-transformer/>
- <https://arxiv.org/abs/1706.03762>
- <https://huggingface.co/docs/transformers/tasks/language_modeling>
- <https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html>
- <https://huggingface.co/docs/transformers/generation_strategies>
- <https://huggingface.co/docs/transformers/chat_templating>
- <https://www.nist.gov/itl/ai-risk-management-framework>
- <https://huggingface.co/learn/llm-course/>
- <https://huggingface.co/docs/transformers/quantization/overview>
- <https://docs.ollama.com/import>
- <https://docs.ollama.com/>
- <https://github.com/curiousily/AI-Bootcamp/blob/master/28.ollama-quickstart.ipynb>